In [14]:
import os
import pandas as pd
from google.cloud import bigquery

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/Users/kasperhansen/Desktop/Gdelt/keen-diode-493214-v8-959e0b8e9bdd.json'

client = bigquery.Client(project='keen-diode-493214-v8')

query = """
WITH parsed AS (
  SELECT 
    SPLIT(SPLIT(V2Locations, ';')[OFFSET(0)], '#')[OFFSET(1)] as Land,
    TRIM(SPLIT(V2Themes, ';')[OFFSET(0)]) as Emne_Raa,
    SPLIT(TRIM(SPLIT(V2Themes, ';')[OFFSET(0)]), ',')[OFFSET(0)] as Emne,
    DocumentIdentifier
  FROM `gdelt-bq.gdeltv2.gkg_partitioned`
  WHERE _PARTITIONDATE = CURRENT_DATE()
    AND V2Themes LIKE '%ECON_%'
    AND V2Locations IS NOT NULL
)
SELECT 
    Land,
    Emne,
    COUNT(DISTINCT DocumentIdentifier) as Antal_Artikler
FROM parsed
WHERE Land IS NOT NULL AND Land != ''
GROUP BY Land, Emne
ORDER BY Antal_Artikler DESC
LIMIT 100
"""

query_job = client.query(query)
df = query_job.to_dataframe(create_bqstorage_client=False)

print(df)

                                       Land                                Emne  Antal_Artikler
0                                      Iran                      TAX_ECON_PRICE             131
1                                     China                EPU_ECONOMY_HISTORIC             126
2                                    German                TAX_ETHNICITY_GERMAN             124
3                                      Iran                       ARMEDCONFLICT             118
4                             United States                EPU_ECONOMY_HISTORIC             116
..                                      ...                                 ...             ...
95                                    China  WB_1921_PRIVATE_SECTOR_DEVELOPMENT              33
96                                     Iran                           MEDIA_MSM              33
97                                 American                EPU_ECONOMY_HISTORIC              33
98                                     I